<a href="https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!git clone https://github.com/abdulmanan2728/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 351 (delta 143), reused 93 (delta 93), pack-reused 178 (from 2)
Receiving objects: 100% (351/351), 1.93 MiB | 12.04 MiB/s, done.
Resolving deltas: 100% (198/198), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Rebuild the w04 baseline rule, so we can compare on the exact same test rows later
median_ctr_by_tier = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_gap'] = median_ctr_by_tier - df['ctr']
stale_mask = df['freshness_tier'].isin(['31-90', '91-180'])
low_ctr_mask = df['ctr_gap'] > 0
df['baseline_flagged'] = (stale_mask & low_ctr_mask).astype(int)

print(df.shape)

(30000, 47)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

**Method: Random Forest classifier.** Per this week's toolkit, decline isn't driven by
one clean signal — it's a tangled combination of staleness, CTR behavior, and content
characteristics, which is exactly the case Random Forest handles well: it captures
nonlinear interactions between features without needing them hand-specified, and its
feature importances give an interpretable read on what's actually driving the score —
useful for turning into reason codes later. Logistic Regression was a candidate too
(more interpretable coefficients), but Random Forest better matches the "tangled,
shifting signals" framing from Week 2.

Before picking features, I checked each numeric candidate against `trend_pct` for
leakage — any column strongly correlated with the label's own source is excluded,
since that would mean the model is just re-deriving the label rather than learning a
real pattern.

In [12]:
candidate_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier_order', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]

leak_corr = df[candidate_cols].corrwith(df['trend_pct']).abs().sort_values(ascending=False)
print(leak_corr)

# drop anything suspiciously close to the label's own source column
LEAK_THRESHOLD = 0.9
leaky_cols = leak_corr[leak_corr > LEAK_THRESHOLD].index.tolist()
feature_cols = [c for c in candidate_cols if c not in leaky_cols]

print(f"\ndropped as leaky (corr > {LEAK_THRESHOLD} with trend_pct): {leaky_cols}")
print(f"remaining features: {feature_cols}")


impressions_last_30d      0.096737
avg_position              0.047248
days_with_impressions     0.031776
impressions_90d           0.024187
competition               0.014426
days_since_last_update    0.014230
impressions_prev_30d      0.010279
word_count                0.009211
engagement_rate           0.008412
ctr                       0.007968
sessions_last_30d         0.007872
days_with_sessions        0.007596
char_count                0.006969
clicks_last_30d           0.006346
sessions_prev_30d         0.005335
scroll_events_90d         0.004865
cpc                       0.004839
clicks_prev_30d           0.004309
ai_traffic_pct            0.003824
users_90d                 0.003704
ai_sessions_90d           0.003465
sessions_90d              0.003162
search_volume             0.002515
scroll_rate               0.001868
clicks_90d                0.001778
engaged_sessions_90d      0.001598
pageviews_90d             0.001189
content_age_days          0.000712
age_tier_order      

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

Grouped by `client_id`, not random. A random split would let the same client's pages
appear in both train and test, letting the model partly memorize client-specific
patterns rather than learning something that generalizes to a client it's never seen —
this is the exact leakage lesson from Week 3's `GroupShuffleSplit` comparison
(0.865 random-split accuracy vs. 0.833 honest grouped-split accuracy on the warehouse
data). Same principle applies here: the split must respect client boundaries to mean
anything.

In [13]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

X = df[feature_cols]
y = df['is_declining']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# confirm no client appears in both sides
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"client overlap between train and test: {len(train_clients & test_clients)}")
print(f"train rows: {len(X_train)}, test rows: {len(X_test)}")

client overlap between train and test: 0
train rows: 22885, test rows: 7115


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

Same test rows, same metric (precision on the "declining" class, per Week 2's success
metric) as the Week-4 baseline rule.

In [14]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

# baseline predictions on the exact same test rows
baseline_preds = df.loc[X_test.index, 'baseline_flagged']

comparison = pd.DataFrame({
    'method': ['baseline_rule', 'random_forest'],
    'precision': [
        precision_score(y_test, baseline_preds),
        precision_score(y_test, model_preds),
    ],
    'recall': [
        recall_score(y_test, baseline_preds),
        recall_score(y_test, model_preds),
    ],
    'f1': [
        f1_score(y_test, baseline_preds),
        f1_score(y_test, model_preds),
    ],
})
print(comparison)
print(f"\nbase rate on test set: {y_test.mean():.3f}")

          method  precision    recall        f1
0  baseline_rule   0.524217  0.100136  0.168152
1  random_forest   0.805353  0.900680  0.850353

base rate on test set: 0.517


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [15]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(10))

# where does the model get it wrong?
test_df = df.loc[X_test.index].copy()
test_df['pred'] = model_preds
test_df['actual'] = y_test

false_positives = test_df[(test_df['pred'] == 1) & (test_df['actual'] == 0)]
false_negatives = test_df[(test_df['pred'] == 0) & (test_df['actual'] == 1)]

print(f"\nfalse positives: {len(false_positives)} ({len(false_positives)/len(test_df):.3f} of test set)")
print(f"false negatives: {len(false_negatives)} ({len(false_negatives)/len(test_df):.3f} of test set)")

print("\nfalse positive avg impressions vs overall avg:")
print(f"  FP: {false_positives['impressions_90d'].mean():.0f}, overall: {test_df['impressions_90d'].mean():.0f}")

impressions_prev_30d     0.211458
impressions_last_30d     0.157783
impressions_90d          0.080401
avg_position             0.061751
days_with_impressions    0.046095
content_age_days         0.043111
word_count               0.034916
char_count               0.029251
sessions_last_30d        0.028634
clicks_last_30d          0.025514
dtype: float64

false positives: 800 (0.112 of test set)
false negatives: 365 (0.051 of test set)

false positive avg impressions vs overall avg:
  FP: 2160, overall: 3728


## 4. Errors and interpretation

**What the model leans on:** `impressions_prev_30d` (0.211) and `impressions_last_30d`
(0.158) dominate — together nearly 37% of total importance — followed by
`impressions_90d` (0.080) and `avg_position` (0.062). This makes sense: recent
impression momentum is the most direct available signal for a trend that's fundamentally
about impressions changing. `content_age_days` and `days_with_impressions` contribute
meaningfully too, suggesting page maturity plays a real secondary role.

**Where it's wrong:** false positives (800 rows, 11.2% of test set) outnumber false
negatives (365 rows, 5.1%) more than 2-to-1 — consistent with the model erring toward
flagging borderline cases rather than missing real ones, which fits the Week 2 framing
choice to prioritize precision over recall. False positives average noticeably lower
impressions (2,160) than the test set overall (3,728) — the model is more likely to
mistakenly flag lower-traffic pages, where a given percentage swing in a small number is
noisier and easier to misread as a real decline trend.

**What this means for the review queue:** low-traffic pages flagged by the model deserve
slightly less confidence than high-traffic ones — worth carrying into the Week 7 action
playbook as a confidence qualifier, not just a binary flag.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.